# 🐍 TD02 — Python Foundations & Data Exploration (CSV / JSON)

## 0. Environment Setup & Project Path Resolution

In [16]:
from pathlib import Path
import csv
import json

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = next(p for p in [CURRENT_DIR, *CURRENT_DIR.parents] if (p / "data" / "sample").exists())
DATA_DIR = PROJECT_ROOT / "data" / "sample"

print(f"Project root resolved to : {PROJECT_ROOT}")
print(f"Data directory           : {DATA_DIR}")
print(f"instruments.json exists  : {(DATA_DIR / 'instruments.json').exists()}")
print(f"prices.csv exists        : {(DATA_DIR / 'prices.csv').exists()}")

Project root resolved to : /workspaces/esilv-marketpulse-g05-t02
Data directory           : /workspaces/esilv-marketpulse-g05-t02/data/sample
instruments.json exists  : True
prices.csv exists        : True


## 1. Python Foundations Refresher: Lists, Dictionaries & Comprehensions

In [17]:
sample_instrument = {
    "ticker": "AAPL",
    "name": "Apple Inc.",
    "currency": "USD",
    "is_active": True
}

print("Dictionary keys   :", list(sample_instrument.keys()))
print("Access by key     :", sample_instrument["name"])
print("Safe get with def :", sample_instrument.get("exchange", "UNKNOWN"))

Dictionary keys   : ['ticker', 'name', 'currency', 'is_active']
Access by key     : Apple Inc.
Safe get with def : UNKNOWN


In [18]:
mock_rows = [
    {"date": "2026-09-01", "ticker": "AAPL", "close": "250.00"},
    {"date": "2026-09-01", "ticker": "SP500", "close": "6600.00"},
    {"date": "2026-09-02", "ticker": "AAPL", "close": "251.30"},
    {"date": "2026-09-02", "ticker": "SP500", "close": "6612.00"},
]

aapl_rows_loop = []
for row in mock_rows:
    if row["ticker"] == "AAPL":
        aapl_rows_loop.append(row)

aapl_rows_comp = [row for row in mock_rows if row["ticker"] == "AAPL"]

print(f"Loop filter count : {len(aapl_rows_loop)}")
print(f"Comp filter count : {len(aapl_rows_comp)}")
assert aapl_rows_loop == aapl_rows_comp, "Both patterns must yield identical results!"

Loop filter count : 2
Comp filter count : 2


## 2. Inspecting and Loading JSON (`instruments.json`)

In [19]:
def load_instruments(filepath=DATA_DIR / "instruments.json"):
    with open(filepath, mode="r", encoding="utf-8") as file:
        return json.load(file)

instruments = load_instruments()
print("Type of loaded JSON:", type(instruments))
print("Top-level keys     :", list(instruments.keys()))

Type of loaded JSON: <class 'dict'>
Top-level keys     : ['instrument', 'benchmark']


In [20]:
instrument = instruments["instrument"]
benchmark = instruments["benchmark"]

print("--- Primary Instrument ---")
print(f"Ticker   : {instrument['ticker']}")
print(f"Name     : {instrument['name']}")
print(f"Currency : {instrument['currency']}")
print(f"Market   : {instrument['market']}")

print("\n--- Benchmark Index ---")
print(f"Ticker   : {benchmark['ticker']}")
print(f"Name     : {benchmark['name']}")
print(f"Currency : {benchmark['currency']}")
print(f"Market   : {benchmark['market']}")

--- Primary Instrument ---
Ticker   : AAPL
Name     : Apple Inc.
Currency : USD
Market   : NASDAQ

--- Benchmark Index ---
Ticker   : SP500
Name     : S&P 500
Currency : USD
Market   : US


## 3. Inspecting and Loading CSV (`prices.csv`)

In [21]:
def load_prices(filepath=DATA_DIR / "prices.csv"):
    with open(filepath, mode="r", encoding="utf-8") as file:
        return list(csv.DictReader(file))

prices = load_prices()
print(f"Total records loaded : {len(prices)}")
print(f"Type of prices       : {type(prices)}")
print(f"Type of each row     : {type(prices[0])}")
print("\nFirst observation sample:")
for key, val in prices[0].items():
    print(f"  {key:8s} -> {val} (type: {type(val).__name__})")

Total records loaded : 42
Type of prices       : <class 'list'>
Type of each row     : <class 'dict'>

First observation sample:
  date     -> 2026-09-01 (type: str)
  ticker   -> AAPL (type: str)
  open     -> 249.20 (type: str)
  high     -> 251.50 (type: str)
  low      -> 248.00 (type: str)
  close    -> 250.00 (type: str)
  volume   -> 38000000 (type: str)


### ⚠️ The Critical CSV Trap: Values Arrive as Text (`str`)!

In [22]:
raw_close = prices[0]["close"]

print(f"String concatenation : raw_close + '10' = {raw_close + '10'}  <-- INCORRECT!")

converted_close = float(raw_close)
print(f"Numerical addition   : converted_close + 10 = {converted_close + 10.0:.2f}  <-- CORRECT!")

String concatenation : raw_close + '10' = 250.0010  <-- INCORRECT!
Numerical addition   : converted_close + 10 = 260.00  <-- CORRECT!


## 4. Filtering Observations by Ticker

In [23]:
def filter_prices(prices, ticker):
    return [row for row in prices if row["ticker"] == ticker]

instrument_prices = filter_prices(prices, instrument["ticker"])
benchmark_prices = filter_prices(prices, benchmark["ticker"])

print(f"Instrument observations ({instrument['ticker']}) : {len(instrument_prices)}")
print(f"Benchmark observations  ({benchmark['ticker']})  : {len(benchmark_prices)}")

assert len(instrument_prices) == 21, "Expected 21 observations for AAPL"
assert len(benchmark_prices) == 21, "Expected 21 observations for SP500"

Instrument observations (AAPL) : 21
Benchmark observations  (SP500)  : 21


## 5. First & Last Close Helpers

In [24]:
def get_first_close(prices):
    if not prices:
        return 0.0
    return float(prices[0]["close"])

def get_last_close(prices):
    if not prices:
        return 0.0
    return float(prices[-1]["close"])

aapl_first = get_first_close(instrument_prices)
aapl_last = get_last_close(instrument_prices)
print(f"AAPL  -> First Close: {aapl_first:.2f} | Last Close: {aapl_last:.2f}")

sp_first = get_first_close(benchmark_prices)
sp_last = get_last_close(benchmark_prices)
print(f"SP500 -> First Close: {sp_first:.2f} | Last Close: {sp_last:.2f}")

assert aapl_first == 250.00, f"Expected 250.00, got {aapl_first}"
assert aapl_last == 266.20, f"Expected 266.20, got {aapl_last}"
assert sp_first == 6600.00, f"Expected 6600.00, got {sp_first}"
assert sp_last == 6742.00, f"Expected 6742.00, got {sp_last}"

AAPL  -> First Close: 250.00 | Last Close: 266.20
SP500 -> First Close: 6600.00 | Last Close: 6742.00


## 6. Reusable Market Summary

In [25]:
def display_market_summary(asset, prices, show_currency=True):
    first_close = get_first_close(prices)
    last_close = get_last_close(prices)
    currency_suffix = f" {asset.get('currency', '')}" if show_currency else ""

    print(f"{asset['ticker']} - {asset['name']}")
    print(f"Observations : {len(prices)}")
    print(f"First close  : {first_close:.2f}{currency_suffix}")
    print(f"Last close   : {last_close:.2f}{currency_suffix}")

print("=== Test Display ===\n")
display_market_summary(instrument, instrument_prices, show_currency=True)
print()
display_market_summary(benchmark, benchmark_prices, show_currency=False)

=== Test Display ===

AAPL - Apple Inc.
Observations : 21
First close  : 250.00 USD
Last close   : 266.20 USD

SP500 - S&P 500
Observations : 21
First close  : 6600.00
Last close   : 6742.00


## 7. Complete Interactive Prototype (Target `src/main.py`)

In [26]:
def run_td02_market_snapshot():
    instruments = load_instruments()
    prices = load_prices()

    instrument = instruments["instrument"]
    benchmark = instruments["benchmark"]

    instrument_prices = filter_prices(prices, instrument["ticker"])
    benchmark_prices = filter_prices(prices, benchmark["ticker"])

    print("=== MarketPulse ===")
    print()
    print("Market configuration")
    print("Period   : 1 month")
    print("Interval : Daily")
    print()
    print("Instrument")
    display_market_summary(instrument, instrument_prices, show_currency=True)
    print()
    print("Benchmark")
    display_market_summary(benchmark, benchmark_prices, show_currency=False)

run_td02_market_snapshot()

=== MarketPulse ===

Market configuration
Period   : 1 month
Interval : Daily

Instrument
AAPL - Apple Inc.
Observations : 21
First close  : 250.00 USD
Last close   : 266.20 USD

Benchmark
SP500 - S&P 500
Observations : 21
First close  : 6600.00
Last close   : 6742.00
